# keyboard_config

> Shared keyboard navigation configuration for the combined Phase 2 step

In [ ]:
#| default_exp components.keyboard_config

In [ ]:
#| export
from typing import Any, Tuple

from fasthtml.common import Details, Summary, Div

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.components.data_display.collapse import (
    collapse, collapse_title, collapse_content, collapse_modifiers
)

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.core.base import combine_classes

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.hints import render_keyboard_hints
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system

# Card stack library
from cjm_fasthtml_card_stack.keyboard.actions import build_card_stack_url_map

# Decomposition-specific keyboard config (building blocks)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.keyboard_config import (
    SD_DECOMP_ENTER_SPLIT_BTN, SD_DECOMP_EXIT_SPLIT_BTN,
    SD_DECOMP_SPLIT_BTN, SD_DECOMP_MERGE_BTN, SD_DECOMP_UNDO_BTN,
    create_decomp_kb_parts,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS, DECOMP_CS_BTN_IDS,
    DECOMP_TS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls

## Keyboard Hints Component

Generic collapsible keyboard hints renderer. Works with any `ZoneManager`
instance, wrapping the library's `render_keyboard_hints` in a DaisyUI collapse.

In [ ]:
#| export
def render_keyboard_hints_collapsible(
    manager:ZoneManager,  # Keyboard zone manager with actions configured
    container_id:str="sd-keyboard-hints",  # HTML ID for the hints container
    include_zone_switch:bool=False,  # Whether to include zone switch hints
) -> Any:  # Collapsible keyboard hints component
    """Render keyboard shortcut hints in a collapsible DaisyUI collapse."""
    hints = render_keyboard_hints(
        manager,
        include_navigation=True,
        include_zone_switch=include_zone_switch,
        badge_style="outline",
        container_id=container_id,
        use_icons=False
    )

    return Details(
        Summary(
            "Keyboard Shortcuts",
            cls=combine_classes(collapse_title, font_size.sm, font_weight.medium)
        ),
        Div(
            hints,
            cls=collapse_content
        ),
        cls=combine_classes(collapse, collapse_modifiers.arrow, bg_dui.base_200)
    )

## Keyboard System Builder

Builds the complete keyboard system (manager + rendered system) for the decomposition
column. Returns both the manager (needed for hints rendering) and the system
(needed for scripts/inputs/buttons in the column body).

When alignment is added, this will expand to a combined builder that creates
both decomposition and alignment zones in a single shared `ZoneManager`.

In [ ]:
#| export
def build_decomp_kb_system(
    urls:DecompUrls,  # URL bundle for decomposition routes
) -> Tuple[ZoneManager, Any]:  # (keyboard manager, rendered keyboard system)
    """Build the complete keyboard system for the decomposition column."""
    # Get decomp-specific building blocks
    decomp_zone, decomp_actions, decomp_modes = create_decomp_kb_parts(
        ids=DECOMP_CS_IDS,
        button_ids=DECOMP_CS_BTN_IDS,
        config=DECOMP_CS_CONFIG,
    )

    # Assemble into ZoneManager (will add alignment zone here later)
    kb_manager = ZoneManager(
        zones=(decomp_zone,),
        actions=decomp_actions,
        modes=decomp_modes,
        prev_zone_key="",
        next_zone_key="",
        state_hidden_inputs=True,
    )

    # Build URL/target/include/swap maps for the keyboard system
    include_selector = (
        f"#{DECOMP_CS_IDS.focused_index_input}, "
        f"#{DECOMP_TS_IDS.anchor_input}"
    )

    url_map = {
        **build_card_stack_url_map(DECOMP_CS_BTN_IDS, urls.card_stack),
        SD_DECOMP_ENTER_SPLIT_BTN: urls.enter_split,
        SD_DECOMP_EXIT_SPLIT_BTN: urls.exit_split,
        SD_DECOMP_SPLIT_BTN: urls.split,
        SD_DECOMP_MERGE_BTN: urls.merge,
        SD_DECOMP_UNDO_BTN: urls.undo,
    }

    card_stack_target = f"#{DECOMP_CS_IDS.card_stack}"
    target_map = {btn_id: card_stack_target for btn_id in url_map}
    include_map = {btn_id: include_selector for btn_id in url_map}
    swap_map = {btn_id: "none" for btn_id in url_map}

    kb_system = render_keyboard_system(
        kb_manager,
        url_map=url_map,
        target_map=target_map,
        include_map=include_map,
        swap_map=swap_map,
        show_hints=False,
        include_state_inputs=True,
    )

    return kb_manager, kb_system

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()